# Iteración, recursos y concurrencia

**Unidad 1 · Sesión 3**

Generadores, context managers y concurrencia con `asyncio`.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Generadores y consumo bajo demanda |
| 2 | `with`, decoradores y `@contextmanager` |
| 3 | `async`/`await` y tareas concurrentes |
| 4 | Tiempos límite y limpieza |

Al terminar, podrás reconocer cuándo conviene procesar valores uno a uno,
mantener un recurso abierto solo durante su uso y solapar esperas independientes.

## 1. Generadores y consumo bajo demanda

Una lista conserva sus elementos. Un **iterador** mantiene la posición de un
recorrido; `next()` solicita el siguiente valor. Cuando se agota, no vuelve al
inicio. Una función con `yield` crea un generador: produce un valor cuando el
consumidor lo pide y continúa desde donde se quedó.

In [1]:
values = iter([18.0, 27.0, 32.0])
print(next(values))
print(list(values))
print(list(values))  # The iterator is exhausted.

18.0
[27.0, 32.0]
[]


In [2]:
from collections.abc import Iterator


def readings(values: list[float]) -> Iterator[float]:
    for value in values:
        print(f"Reading {value}")
        yield value


stream = readings([18.0, 27.0, 32.0])
print("Generator created")
print(next(stream))
print(list(stream))

Generator created
Reading 18.0
18.0
Reading 27.0
Reading 32.0
[27.0, 32.0]


Crear `stream` no ejecutó el cuerpo de `readings`. El primer `next()` produjo
un solo valor; `list(stream)` consumió el resto. Esto permite recorrer datos
sin construir antes una lista con todos los resultados. Si el consumidor usa
`list()`, entonces sí materializa lo que queda.

### Ejercicio 1 · Números pares bajo demanda

Escribe `even_numbers(stop: int)`, una función generadora que produzca los
números pares desde cero hasta antes de `stop`. Comprueba que
`list(even_numbers(7))` sea `[0, 2, 4, 6]` y que `list(even_numbers(0))`
sea una lista vacía. Usa `yield`; no construyas una lista dentro de la función.

In [3]:
def even_numbers(stop: int) -> Iterator[int]:
    for number in range(0, stop, 2):
        yield number


assert list(even_numbers(7)) == [0, 2, 4, 6]
assert list(even_numbers(0)) == []
print("Ejercicio 1 completado")


Ejercicio 1 completado


## 2. Recursos con `with`

Un archivo debe cerrarse al terminar de usarlo. `with` delimita ese periodo:
al salir del bloque, Python llama al cierre del recurso, también si dentro
ocurre un error. El archivo se puede recorrer línea por línea.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory


with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")

    with path.open(encoding="utf-8") as file:
        first_line = next(file).strip()
        print(first_line)

    assert file.closed

north;18.0


### Salida por error

El cierre no depende de que el bloque termine normalmente. El error puede
atenderse fuera de `with` y el archivo ya estará cerrado.

In [5]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("invalid\n", encoding="utf-8")

    try:
        with path.open(encoding="utf-8") as file:
            raise ValueError("Invalid reading")
    except ValueError as error:
        print(error)

    assert file.closed

Invalid reading


### Qué es un decorador

Un decorador recibe una función y devuelve otra que añade comportamiento.
La forma `@announce` aplica `announce` a la función definida debajo. Aquí
solo observamos esa idea; no hace falta crear una biblioteca de decoradores.


In [6]:
def announce(function):
    def wrapper():
        print("Starting")
        return function()

    return wrapper


@announce
def show_message():
    print("Reading complete")


show_message()

Starting
Reading complete


### Crear un context manager con `@contextmanager`

`@contextmanager` permite escribir un context manager sin definir una clase.
La función llega hasta `yield` al entrar en `with`; el bloque `finally` se
ejecuta al salir. En esta demostración, el recurso es un archivo en memoria.

In [7]:
from contextlib import contextmanager
from io import StringIO


@contextmanager
def open_text(text: str) -> Iterator[StringIO]:
    file = StringIO(text)
    try:
        yield file
    finally:
        file.close()


with open_text("north\nsouth\n") as file:
    print(next(file).strip())

assert file.closed

north


### Ejercicio 2 · Cierre de un archivo

Crea un archivo temporal con dos líneas. Ábrelo con `with`, lee solo la primera
y comprueba que esté cerrado después del bloque. Luego provoca un `ValueError`
dentro de otro bloque `with`, atiéndelo fuera del bloque y comprueba de nuevo
que el archivo quedó cerrado.

In [8]:
with TemporaryDirectory() as directory:
    exercise_path = Path(directory) / "two_lines.txt"
    exercise_path.write_text("first line\nsecond line\n", encoding="utf-8")

    with exercise_path.open(encoding="utf-8") as exercise_file:
        assert next(exercise_file).strip() == "first line"
        assert not exercise_file.closed
    assert exercise_file.closed

    try:
        with exercise_path.open(encoding="utf-8") as error_file:
            raise ValueError("Intentional exercise error")
    except ValueError as error:
        assert str(error) == "Intentional exercise error"
    else:
        raise AssertionError("El ValueError debía propagarse fuera del bloque with")

    assert error_file.closed

print("Ejercicio 2 completado")


Ejercicio 2 completado


## 3. Concurrencia con `asyncio`

Primero simulamos una lectura **síncrona**: `time.sleep` bloquea el hilo durante
los segundos indicados. Dos llamadas consecutivas de 3 y 2 segundos tardan
aproximadamente 5 segundos. Los mensajes permiten observar cuándo comienza y
termina cada operación.

In [9]:
import asyncio
import time
from time import perf_counter


def fetch_reading_sync(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    time.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


start = perf_counter()
print(fetch_reading_sync("north", 3))
print(fetch_reading_sync("south", 2))
print(f"Synchronous: {perf_counter() - start:.1f} s")

Requesting north


Received north
('north', 18.0)
Requesting south


Received south
('south', 27.0)
Synchronous: 5.0 s


### La versión asíncrona

`async def` define una corrutina. `await asyncio.sleep()` suspende esa corrutina
y permite que otras tareas avancen. Cambiar solo `def` por `async def` no haría
que `time.sleep` dejara de bloquear: también necesitamos una espera compatible.

Dos `await` consecutivos siguen siendo **secuenciales**. En Jupyter y Colab usamos
`await` directamente en una celda.

In [10]:
async def fetch_reading(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    await asyncio.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


start = perf_counter()
north = await fetch_reading("north", 3)
south = await fetch_reading("south", 2)
print(north, south, f"Sequential: {perf_counter() - start:.1f} s")

Requesting north


Received north
Requesting south


Received south
('north', 18.0) ('south', 27.0) Sequential: 5.0 s


### Adaptar una función síncrona con `to_thread`

Si una biblioteca de E/S solo ofrece una función síncrona, `asyncio.to_thread`
puede ejecutarla en otro hilo. Este wrapper conserva la función original y
permite esperar su resultado con `await`. Pasamos la función **sin llamarla**,
seguida de sus argumentos. Un único `await` no hace que la lectura dure menos.

Esto sirve para esperas bloqueantes de E/S. No garantiza acelerar cálculos de
Python, y cancelar la espera no detiene automáticamente un hilo que ya trabaja.

In [11]:
async def fetch_reading_in_thread(station: str, delay: float) -> tuple[str, float]:
    return await asyncio.to_thread(fetch_reading_sync, station, delay)


print(await fetch_reading_in_thread("north", 2))

Requesting north


Received north
('north', 18.0)


### Una tarea en segundo plano con `create_task`

`create_task()` programa una corrutina para que avance mientras la corrutina
principal hace otro trabajo y cede el control. Conservamos la tarea en una
variable y al final usamos `await` para obtener su resultado y observar errores.
Aquí la lectura continúa durante la pausa de un segundo de la tarea principal.
No es un trabajo persistente: depende de que el event loop siga activo.

In [12]:
reading_task = asyncio.create_task(fetch_reading("north", 3))
print("Main task continues")
await asyncio.sleep(1)
print("Main task is ready to use the reading")
print(await reading_task)

Main task continues
Requesting north


Main task is ready to use the reading


Received north
('north', 18.0)


### Reunir resultados con `gather`

`gather()` programa las corrutinas y devuelve los resultados en el orden en que
las pasamos, aunque terminen en otro orden. Las esperas de 3 y 2 segundos se
solapan: el total se acerca a 3 segundos.

In [13]:
start = perf_counter()
north, south = await asyncio.gather(
    fetch_reading("north", 3),
    fetch_reading("south", 2),
)
print(north, south, f"Gather: {perf_counter() - start:.1f} s")

Requesting north
Requesting south


Received south


Received north
('north', 18.0) ('south', 27.0) Gather: 3.0 s


### Delimitar las tareas con `TaskGroup`

`TaskGroup` espera sus tareas al salir de `async with`. Conservamos cada objeto
tarea para consultar después su resultado. Si una tarea falla con una excepción
que no es de cancelación, el grupo cancela las restantes y espera su limpieza.

En cambio, `gather` con sus opciones predeterminadas propaga la primera excepción,
pero las otras tareas pueden continuar. Aquí ambos ejemplos tienen éxito; la
diferencia está en cómo organizan su finalización y sus errores.

In [14]:
start = perf_counter()
async with asyncio.TaskGroup() as group:
    north_task = group.create_task(fetch_reading("north", 3))
    south_task = group.create_task(fetch_reading("south", 2))
print(north_task.result(), south_task.result())
print(f"TaskGroup: {perf_counter() - start:.1f} s")

Requesting north
Requesting south


Received south


Received north
('north', 18.0) ('south', 27.0)
TaskGroup: 3.0 s


Los tiempos son aproximados. Observa que ambas solicitudes empiezan antes de
que llegue la primera respuesta. La concurrencia permite solapar esperas
independientes; no acelera por sí sola un cálculo intensivo.

### Ejercicio 3 · Dos lecturas concurrentes

Crea dos tareas con `TaskGroup`: `fetch_reading("north", 3)` y
`fetch_reading("west", 1)`. Después del bloque, comprueba con `assert`
los dos resultados. Explica por qué `west` puede anunciarse primero aunque
la tarea de `north` se haya creado antes.

In [15]:
completion_order: list[str] = []


async def tracked_reading(station: str, delay: float) -> tuple[str, float]:
    result = await fetch_reading(station, delay)
    completion_order.append(station)
    return result


async with asyncio.TaskGroup() as exercise_group:
    exercise_north = exercise_group.create_task(tracked_reading("north", 3))
    exercise_west = exercise_group.create_task(tracked_reading("west", 1))

assert exercise_north.result() == ("north", 18.0)
assert exercise_west.result() == ("west", 32.0)
assert completion_order == ["west", "north"]
# Ambas tareas avanzan durante la espera; west termina primero porque su demora es menor.
print("Ejercicio 3 completado", completion_order)


Requesting north
Requesting west


Received west


Received north
Ejercicio 3 completado ['west', 'north']


## 4. Tiempo límite y limpieza

Una operación puede tardar demasiado. `asyncio.timeout()` establece un tiempo
máximo para el bloque. Si se agota, se produce `TimeoutError`. La corrutina
puede usar `finally` para limpiar su estado incluso cuando se cancela la espera.

In [16]:
events: list[str] = []


async def wait_for_update() -> None:
    events.append("started")
    try:
        await asyncio.Event().wait()
    finally:
        events.append("cleaned up")


try:
    async with asyncio.timeout(2):
        await wait_for_update()
except TimeoutError:
    print("Deadline reached")

assert events == ["started", "cleaned up"]
print(events)

Deadline reached
['started', 'cleaned up']


### Ejercicio 4 · Comprobar la limpieza

Escribe una corrutina que añada `"started"` a una lista, espere 3 segundos
y añada `"cleaned up"` en `finally`. Llámala dentro de
`asyncio.timeout(1)`, atiende `TimeoutError` y comprueba que ambos mensajes
quedaron en la lista. Después prueba una espera de 1 segundo con un límite de 3 segundos.

In [17]:
async def cleanup_after(delay: float, log: list[str]) -> None:
    log.append("started")
    try:
        await asyncio.sleep(delay)
    finally:
        log.append("cleaned up")


timeout_events: list[str] = []
try:
    async with asyncio.timeout(1):
        await cleanup_after(3, timeout_events)
except TimeoutError:
    pass
else:
    raise AssertionError("La espera de 3 segundos debía exceder el límite")

assert timeout_events == ["started", "cleaned up"]

completed_events: list[str] = []
async with asyncio.timeout(3):
    await cleanup_after(1, completed_events)
assert completed_events == ["started", "cleaned up"]
print("Ejercicio 4 completado")


Ejercicio 4 completado


## Cierre

- Un generador produce valores cuando se solicitan; consumirlo por completo
  puede volver a ocupar memoria para guardar los resultados.
- `with` delimita el uso de un recurso; `@contextmanager` permite definir
  esa entrada y salida mediante una función.
- `asyncio` ayuda a solapar esperas independientes. Usa un tiempo límite
  cuando una espera no deba continuar indefinidamente.

En un archivo `.py` se inicia el event loop desde una función principal con
`asyncio.run(main())`; en esta notebook usamos `await` directamente.

## Referencias

- [Tutorial de Python: generadores](https://docs.python.org/3/tutorial/classes.html#generators).
- [Python: `contextlib`](https://docs.python.org/3/library/contextlib.html).
- [Python: corrutinas y tareas](https://docs.python.org/3/library/asyncio-task.html).
- [Python: ejecutar `asyncio`](https://docs.python.org/3/library/asyncio-runner.html).